# Credit Card Fraud Detection - Modeling

This notebook builds and evaluates machine learning models for predicting fraudulent credit card transactions.

Features selected based on EDA:
- Transaction amount
- Merchant category
- Hour of transaction
- Day of week
- Customer age
- Customer-to-merchant distance

After restarting the kernel, run sections 1-4 and the **Load Saved Train 6 Model** cell below, then jump to **Train 7**. These setup cells do not train models. The historical experiment cells retain their training code and saved outputs; skip them when resuming. Paths assume this notebook is run from `notebooks/`.

## 1. Imports

In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)

## 2. Load Dataset

In [44]:
df = pd.read_csv("../data/fraudTrain.csv")

df.shape

(1296675, 23)

## 3. Feature Engineering

Create the features identified during exploratory data analysis.

In [45]:
df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"]
)

df["dob"] = pd.to_datetime(df["dob"])

In [46]:
df["hour"] = df["trans_date_trans_time"].dt.hour
df["day_of_week"] = df["trans_date_trans_time"].dt.day_name()

In [47]:
df["age"] = (
    (df["trans_date_trans_time"] - df["dob"]).dt.days / 365.25
).astype(int)

In [48]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [49]:
df["distance_km"] = haversine(
    df["lat"],
    df["long"],
    df["merch_lat"],
    df["merch_long"]
)

In [50]:
df["is_late_night"] = (
    (df["hour"] >= 22) | (df["hour"] <= 3)
).astype(int)

## 4. Temporal Train/Test Setup

Use the oldest 80% of transactions for training and the newest 20% for testing, matching Train 6. Sort into `df_sorted` so `df` keeps its original row order for the historical random-split experiments.

In [51]:
df_sorted = df.sort_values("trans_date_trans_time").reset_index(drop=True)

In [52]:
features_v6 = [
    "amt",
    "category",
    "hour",
    "day_of_week",
    "age",
    "distance_km",
    "is_late_night"
]

features = features_v6.copy()
X = df_sorted[features]
y = df_sorted["is_fraud"]

split_index = int(len(df_sorted) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Keep Train 6's original names available for its evaluation cells.
X_train_v6 = X_train
X_test_v6 = X_test
y_train_v6 = y_train
y_test_v6 = y_test

In [53]:
numeric_features_rf = [
    "amt",
    "hour",
    "age",
    "distance_km",
    "is_late_night"
]

categorical_features = ["category", "day_of_week"]

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features_rf),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

### Load Saved Train 6 Model

Load the already fitted pipeline, including its fitted preprocessing. No `.fit()` call is needed. `preprocessor_rf` above is an unfitted configuration for a new experiment; `model_rf_temporal` is the saved trained model.

In [55]:
model_rf_temporal = joblib.load(
    "../models/random_forest_temporal.pkl"
)

print("Train 6 model loaded!")

Train 6 model loaded!


## Historical Experiments: Train 1-Train 6

Original training code and recorded outputs are preserved below. These cells reuse variables such as `X_train` and `X_test` for their original splits. If you run historical cells, rerun sections 1-4 and reload the saved model before continuing with Train 7. Recorded outputs reflect earlier executions, including cells run out of order.

## Train 1 - Logistic Regression

In [8]:
features = [
    "amt",
    "category",
    "hour",
    "day_of_week",
    "age",
    "distance_km"
]

df[features + ["is_fraud"]].head()

,amt,category,hour,day_of_week,age,distance_km,is_fraud
0,4.97,misc_net,0,Tuesday,30,78.597568,0
1,107.23,grocery_pos,0,Tuesday,40,30.212176,0
2,220.11,entertainment,0,Tuesday,56,108.206083,0
3,45.00,gas_transport,0,Tuesday,51,95.673231,0
4,41.96,misc_pos,0,Tuesday,32,77.556744,0


### Prepare Training Data

Separate the input features from the target variable (`is_fraud`) before training the model.

In [9]:
X = df[features]
y = df["is_fraud"]

In [10]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1296675, 6)
y shape: (1296675,)


### Train/Test Split

Split the dataset into training and testing sets. Stratification is used to preserve the highly imbalanced fraud distribution in both sets.

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [39]:
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining fraud rate:")
print(y_train.value_counts(normalize=True))

print("\nTest fraud rate:")
print(y_test.value_counts(normalize=True))

Training set: (1037340, 7)
Test set: (259335, 7)

Training fraud rate:
is_fraud
0    0.994211
1    0.005789
Name: proportion, dtype: float64

Test fraud rate:
is_fraud
0    0.994212
1    0.005788
Name: proportion, dtype: float64


### Preprocessing

Prepare numerical and categorical features for machine learning.

In [35]:
numeric_features = [
    "amt",
    "hour",
    "age",
    "distance_km"
]

categorical_features = [
    "category",
    "day_of_week"
]

In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [37]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

### Train Logistic Regression Model

Train the preprocessing pipeline and Logistic Regression classifier using the training data.

In [18]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['amt','category','hour','day_of_week','age','distance_km']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining

In [19]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

coefficients = model.named_steps["classifier"].coef_[0]

feature_weights = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

feature_weights = feature_weights.sort_values(
    "coefficient",
    ascending=False
)

feature_weights

,feature,coefficient
8,cat__category_grocery_pos,2.040640
12,cat__category_misc_net,1.495423
6,cat__category_gas_transport,1.334883
15,cat__category_shopping_net,1.032053
1,num__hour,0.943225
0,num__amt,0.370857
2,num__age,0.359726
7,cat__category_grocery_net,0.250317
16,cat__category_shopping_pos,0.071360
3,num__distance_km,-0.001632


In [20]:
y_pred = model.predict(X_test)

In [21]:
print(y_pred[:20])
print(y_test.head(20).values)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [22]:
print("Predicted fraud:", y_pred.sum())
print("Actual fraud:", y_test.sum())

Predicted fraud: 110
Actual fraud: 1501


In [23]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[257729    105]
 [  1496      5]]


In [24]:
print(classification_report(y_test, y_pred, digits=4))

y_prob = model.predict_proba(X_test)[:, 1]
print("PR-AUC:", average_precision_score(y_test, y_prob))

              precision    recall  f1-score   support

           0     0.9942    0.9996    0.9969    257834
           1     0.0455    0.0033    0.0062      1501

    accuracy                         0.9938    259335
   macro avg     0.5198    0.5015    0.5016    259335
weighted avg     0.9887    0.9938    0.9912    259335

PR-AUC: 0.2294662238062145


## Train 2 - Balanced Logistic Regression

In [25]:
model_balanced = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ]
)

In [26]:
model_balanced.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['amt','category','hour','day_of_week','age','distance_km']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining

In [27]:
y_pred_balanced = model_balanced.predict(X_test)

In [28]:
print("Predicted fraud:", y_pred_balanced.sum())
print("Actual fraud:", y_test.sum())

Predicted fraud: 31514
Actual fraud: 1501


In [29]:
cm_balanced = confusion_matrix(y_test, y_pred_balanced)
print(cm_balanced)

[[227458  30376]
 [   363   1138]]


In [30]:
print(classification_report(y_test, y_pred_balanced, digits=4))

y_prob_balanced = model_balanced.predict_proba(X_test)[:, 1]

print(
    "PR-AUC:",
    average_precision_score(y_test, y_prob_balanced)
)

              precision    recall  f1-score   support

           0     0.9984    0.8822    0.9367    257834
           1     0.0361    0.7582    0.0689      1501

    accuracy                         0.8815    259335
   macro avg     0.5173    0.8202    0.5028    259335
weighted avg     0.9928    0.8815    0.9317    259335

PR-AUC: 0.17420925248560126


In [31]:
precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_prob_balanced
)

print("Number of thresholds:", len(thresholds))
print(thresholds[:10])

Number of thresholds: 259290
[0.00139577 0.00140708 0.00141622 0.00141946 0.00142073 0.00142277
 0.00143002 0.00143013 0.00143063 0.00143374]


In [32]:
for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    y_pred_threshold = (y_prob_balanced >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f} | "
        f"Predicted Fraud: {y_pred_threshold.sum()}"
    )

Threshold: 0.5 | Precision: 0.0361 | Recall: 0.7582 | F1: 0.0689 | Predicted Fraud: 31514
Threshold: 0.6 | Precision: 0.0860 | Recall: 0.7488 | F1: 0.1543 | Predicted Fraud: 13065
Threshold: 0.7 | Precision: 0.1921 | Recall: 0.7302 | F1: 0.3042 | Predicted Fraud: 5704
Threshold: 0.8 | Precision: 0.2385 | Recall: 0.6522 | F1: 0.3493 | Predicted Fraud: 4104
Threshold: 0.9 | Precision: 0.2385 | Recall: 0.5057 | F1: 0.3241 | Predicted Fraud: 3183


## Train 3 - Balanced Logistic Regression with Late-Night Feature

In [17]:
numeric_features = [
    "amt",
    "hour",
    "age",
    "distance_km",
    "is_late_night"
]

In [18]:
features = [
    "amt",
    "category",
    "hour",
    "day_of_week",
    "age",
    "distance_km",
    "is_late_night"
]

In [19]:
print(df["is_late_night"].value_counts())
print(
    df.groupby("is_late_night")["is_fraud"].mean() * 100
)

is_late_night
0    991793
1    304882
Name: count, dtype: int64
is_late_night
0    0.115347
1    2.086709
Name: is_fraud, dtype: float64


In [37]:
X = df[features]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [38]:
preprocessor_v3 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [39]:
model_v3 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v3),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ]
)

In [40]:
model_v3.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['amt','category','hour',...,'age','distance_km','is_late_night']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rem

In [41]:
y_pred_v3 = model_v3.predict(X_test)
y_prob_v3 = model_v3.predict_proba(X_test)[:, 1]

print("Predicted fraud:", y_pred_v3.sum())
print("Actual fraud:", y_test.sum())

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_v3))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_v3, digits=4))

print(
    "PR-AUC:",
    average_precision_score(y_test, y_prob_v3)
)

Predicted fraud: 36560
Actual fraud: 1501

Confusion Matrix:
[[222667  35167]
 [   108   1393]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9995    0.8636    0.9266    257834
           1     0.0381    0.9280    0.0732      1501

    accuracy                         0.8640    259335
   macro avg     0.5188    0.8958    0.4999    259335
weighted avg     0.9940    0.8640    0.9217    259335

PR-AUC: 0.25131996612337654


## Train 4 - Balanced Logistic Regression with Amount Flags

In [42]:
df["amt_over_250"] = (df["amt"] >= 250).astype(int)
df["amt_over_500"] = (df["amt"] >= 500).astype(int)

In [43]:
print(
    df.groupby("amt_over_250")["is_fraud"].mean() * 100
)

print(
    df.groupby("amt_over_500")["is_fraud"].mean() * 100
)

amt_over_250
0     0.150690
1    14.586963
Name: is_fraud, dtype: float64
amt_over_500
0     0.301161
1    23.338238
Name: is_fraud, dtype: float64


In [44]:
features = [
    "amt",
    "category",
    "hour",
    "day_of_week",
    "age",
    "distance_km",
    "is_late_night",
    "amt_over_250",
    "amt_over_500"
]

numeric_features = [
    "amt",
    "hour",
    "age",
    "distance_km",
    "is_late_night",
    "amt_over_250",
    "amt_over_500"
]

In [45]:
X = df[features]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [46]:
preprocessor_v4 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [47]:
model_v4 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v4),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ]
)

model_v4.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['amt','category','hour',...,'is_late_night','amt_over_250','amt_over_500']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'`

In [48]:
y_pred_v4 = model_v4.predict(X_test)
y_prob_v4 = model_v4.predict_proba(X_test)[:, 1]

print("Predicted fraud:", y_pred_v4.sum())
print("Actual fraud:", y_test.sum())

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_v4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_v4, digits=4))

print(
    "PR-AUC:",
    average_precision_score(y_test, y_prob_v4)
)

Predicted fraud: 32151
Actual fraud: 1501

Confusion Matrix:
[[227098  30736]
 [    86   1415]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9996    0.8808    0.9365    257834
           1     0.0440    0.9427    0.0841      1501

    accuracy                         0.8811    259335
   macro avg     0.5218    0.9117    0.5103    259335
weighted avg     0.9941    0.8811    0.9315    259335

PR-AUC: 0.2295352827954788


## Train 5 - Random Forest with Random Split

In [49]:
features_v5 = [
    "amt",
    "category",
    "hour",
    "day_of_week",
    "age",
    "distance_km",
    "is_late_night"
]

X = df[features_v5]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [53]:
model_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor_rf),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [54]:
model_rf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['amt','category','hour',...,'age','distance_km','is_late_night']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rem

In [55]:
y_pred_rf = model_rf.predict(X_test)
y_prob_rf = model_rf.predict_proba(X_test)[:, 1]

print("Predicted fraud:", y_pred_rf.sum())
print("Actual fraud:", y_test.sum())

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, digits=4))

print(
    "PR-AUC:",
    average_precision_score(y_test, y_prob_rf)
)

Predicted fraud: 1471
Actual fraud: 1501

Confusion Matrix:
[[257611    223]
 [   253   1248]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9990    0.9991    0.9991    257834
           1     0.8484    0.8314    0.8398      1501

    accuracy                         0.9982    259335
   macro avg     0.9237    0.9153    0.9195    259335
weighted avg     0.9981    0.9982    0.9982    259335

PR-AUC: 0.891235763086833


## Train 6 - Random Forest with Temporal Split

Uses the temporal split and `preprocessor_rf` defined in section 4. The original training cell below is retained for reference; use the saved-model loader above when resuming.

In [58]:
print("Training:", X_train_v6.shape)
print("Test:", X_test_v6.shape)

print("Training fraud rate:", y_train_v6.mean() * 100)
print("Test fraud rate:", y_test_v6.mean() * 100)

print(
    "Training dates:",
    df_sorted["trans_date_trans_time"].iloc[0],
    "to",
    df_sorted["trans_date_trans_time"].iloc[split_index - 1]
)

print(
    "Test dates:",
    df_sorted["trans_date_trans_time"].iloc[split_index],
    "to",
    df_sorted["trans_date_trans_time"].iloc[-1]
)

Training: (1037340, 7)
Test: (259335, 7)
Training fraud rate: 0.5753176393467908
Test fraud rate: 0.5930553145545336
Training dates: 2019-01-01 00:00:18 to 2020-03-06 07:15:17
Test dates: 2020-03-06 07:16:43 to 2020-06-21 12:13:37


In [59]:
model_rf_temporal = Pipeline(
    steps=[
        ("preprocessor", preprocessor_rf),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

model_rf_temporal.fit(X_train_v6, y_train_v6)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['amt','category','hour',...,'age','distance_km','is_late_night']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rem

In [60]:
y_pred_v6 = model_rf_temporal.predict(X_test_v6)
y_prob_v6 = model_rf_temporal.predict_proba(X_test_v6)[:, 1]

print("Predicted fraud:", y_pred_v6.sum())
print("Actual fraud:", y_test_v6.sum())

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_v6, y_pred_v6))

print("\nClassification Report:")
print(classification_report(y_test_v6, y_pred_v6, digits=4))

print(
    "PR-AUC:",
    average_precision_score(y_test_v6, y_prob_v6)
)

Predicted fraud: 1492
Actual fraud: 1538

Confusion Matrix:
[[257571    226]
 [   272   1266]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9989    0.9991    0.9990    257797
           1     0.8485    0.8231    0.8356      1538

    accuracy                         0.9981    259335
   macro avg     0.9237    0.9111    0.9173    259335
weighted avg     0.9981    0.9981    0.9981    259335

PR-AUC: 0.8862019520563971


### Original Model-Saving Cell

Retained from the original experiments. Running this cell writes the model files; it is not needed when resuming from the saved Train 6 model.

In [62]:
# Save Train 5 - Random Forest with random split
joblib.dump(
    model_rf,
    "../models/random_forest_random_split.pkl"
)

# Save Train 6 - Random Forest with temporal split
joblib.dump(
    model_rf_temporal,
    "../models/random_forest_temporal.pkl"
)

print("Both models saved successfully!")

Both models saved successfully!


## Train 7 - Continue Experimenting

After a kernel restart, run sections 1-4 and load the saved Train 6 model above, then continue here. `X_train`, `X_test`, `y_train`, and `y_test` will use the Train 6 temporal split. The cell below only defines the new model.

In [59]:
model_rf_v7 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_rf),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [60]:
model_rf_v7.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['amt','category','hour',...,'age','distance_km','is_late_night']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rem

In [61]:
y_pred_v7 = model_rf_v7.predict(X_test)
y_prob_v7 = model_rf_v7.predict_proba(X_test)[:, 1]

In [62]:
print("Predicted fraud:", y_pred_v7.sum())
print("Actual fraud:", y_test.sum())

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_v7))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_v7, digits=4))

print(
    "PR-AUC:",
    average_precision_score(y_test, y_prob_v7)
)

Predicted fraud: 2513
Actual fraud: 1538

Confusion Matrix:
[[256679   1118]
 [   143   1395]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9994    0.9957    0.9975    257797
           1     0.5551    0.9070    0.6887      1538

    accuracy                         0.9951    259335
   macro avg     0.7773    0.9513    0.8431    259335
weighted avg     0.9968    0.9951    0.9957    259335

PR-AUC: 0.8837685970308667


### Train 7 - Threshold Testing
Test different classification thresholds to examine the tradeoff between fraud precision and recall.

In [63]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

for threshold in thresholds:
    y_pred_threshold = (y_prob_v7 >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    print(f"Threshold: {threshold}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1: {f1:.4f}")
    print(f"Predicted fraud: {y_pred_threshold.sum()}")
    print()

Threshold: 0.5
Precision: 0.5551
Recall: 0.9070
F1: 0.6887
Predicted fraud: 2513

Threshold: 0.6
Precision: 0.6587
Recall: 0.8810
F1: 0.7538
Predicted fraud: 2057

Threshold: 0.7
Precision: 0.7509
Recall: 0.8466
F1: 0.7958
Predicted fraud: 1734

Threshold: 0.8
Precision: 0.8322
Recall: 0.7997
F1: 0.8156
Predicted fraud: 1478

Threshold: 0.9
Precision: 0.9276
Recall: 0.7003
F1: 0.7981
Predicted fraud: 1161



## Train 8 - Hyperparameter Tuning
Use a temporal validation set to compare Random Forest hyperparameters without tuning on the final test set.

In [64]:
df_sorted = df.sort_values(
    "trans_date_trans_time"
).reset_index(drop=True)

X = df_sorted[features]
y = df_sorted["is_fraud"]

train_end = int(len(df_sorted) * 0.70)
validation_end = int(len(df_sorted) * 0.80)

X_train_v8 = X.iloc[:train_end]
y_train_v8 = y.iloc[:train_end]

X_val_v8 = X.iloc[train_end:validation_end]
y_val_v8 = y.iloc[train_end:validation_end]

X_test_v8 = X.iloc[validation_end:]
y_test_v8 = y.iloc[validation_end:]

In [65]:
print("Training:", X_train_v8.shape)
print("Validation:", X_val_v8.shape)
print("Test:", X_test_v8.shape)

print("\nFraud rates:")
print("Training:", y_train_v8.mean())
print("Validation:", y_val_v8.mean())
print("Test:", y_test_v8.mean())

Training: (907672, 7)
Validation: (129668, 7)
Test: (259335, 7)

Fraud rates:
Training: 0.0056419058867079735
Validation: 0.006532066508313539
Test: 0.005930553145545337


In [67]:
parameter_sets = [
    {"max_depth": None, "min_samples_leaf": 1},
    {"max_depth": 20, "min_samples_leaf": 2},
    {"max_depth": 30, "min_samples_leaf": 2},
    {"max_depth": 30, "min_samples_leaf": 5}
]

In [68]:
results_v8 = []

for params in parameter_sets:
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor_rf),
            ("classifier", RandomForestClassifier(
                n_estimators=100,
                max_depth=params["max_depth"],
                min_samples_leaf=params["min_samples_leaf"],
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            ))
        ]
    )

    model.fit(X_train_v8, y_train_v8)

    y_pred = model.predict(X_val_v8)
    y_prob = model.predict_proba(X_val_v8)[:, 1]

    results_v8.append({
        "max_depth": params["max_depth"],
        "min_samples_leaf": params["min_samples_leaf"],
        "precision": precision_score(y_val_v8, y_pred),
        "recall": recall_score(y_val_v8, y_pred),
        "f1": f1_score(y_val_v8, y_pred),
        "pr_auc": average_precision_score(y_val_v8, y_prob)
    })

In [69]:
results_df_v8 = pd.DataFrame(results_v8)
results_df_v8

,max_depth,min_samples_leaf,precision,recall,f1,pr_auc
0,NaN,1,0.877039,0.825266,0.850365,0.905470
1,20.0,2,0.563991,0.920897,0.699552,0.903171
2,30.0,2,0.777426,0.870130,0.821170,0.911299
3,30.0,5,0.612670,0.902007,0.729704,0.904473


## Train 9 - Random Forest Feature Sampling

Test whether changing the number of features considered at each tree split improves validation performance.

In [71]:
parameter_sets_v9 = [
    {
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": 0.5
    },
    {
        "max_depth": 30,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    },
    {
        "max_depth": 30,
        "min_samples_leaf": 2,
        "max_features": 0.5
    }
]

In [72]:
results_v9 = []

for params in parameter_sets_v9:
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor_rf),
            ("classifier", RandomForestClassifier(
                n_estimators=100,
                max_depth=params["max_depth"],
                min_samples_leaf=params["min_samples_leaf"],
                max_features=params["max_features"],
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            ))
        ]
    )

    model.fit(X_train_v8, y_train_v8)

    y_pred = model.predict(X_val_v8)
    y_prob = model.predict_proba(X_val_v8)[:, 1]

    results_v9.append({
        "max_depth": params["max_depth"],
        "min_samples_leaf": params["min_samples_leaf"],
        "max_features": params["max_features"],
        "precision": precision_score(y_val_v8, y_pred),
        "recall": recall_score(y_val_v8, y_pred),
        "f1": f1_score(y_val_v8, y_pred),
        "pr_auc": average_precision_score(y_val_v8, y_prob)
    })

In [74]:
results_df_v9 = pd.DataFrame(results_v9)
results_df_v9

,max_depth,min_samples_leaf,max_features,precision,recall,f1,pr_auc
0,NaN,1,sqrt,0.877039,0.825266,0.850365,0.905470
1,NaN,1,0.5,0.855792,0.854782,0.855286,0.917359
2,30.0,2,sqrt,0.777426,0.870130,0.821170,0.911299
3,30.0,2,0.5,0.724665,0.894923,0.800845,0.914515


## Final Random Forest Model

Train the selected Random Forest configuration on the full first 80% of historical data, then evaluate once on the held-out newest 20%.

In [75]:
final_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor_rf),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_leaf=1,
            max_features=0.5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [76]:
final_rf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['amt','category','hour',...,'age','distance_km','is_late_night']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rem

In [77]:
y_pred_final = final_rf.predict(X_test)
y_prob_final = final_rf.predict_proba(X_test)[:, 1]

In [78]:
print("Predicted fraud:", y_pred_final.sum())
print("Actual fraud:", y_test.sum())

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final, digits=4))

print(
    "PR-AUC:",
    average_precision_score(y_test, y_prob_final)
)

Predicted fraud: 1543
Actual fraud: 1538

Confusion Matrix:
[[257532    265]
 [   260   1278]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9990    0.9990    0.9990    257797
           1     0.8283    0.8309    0.8296      1538

    accuracy                         0.9980    259335
   macro avg     0.9136    0.9150    0.9143    259335
weighted avg     0.9980    0.9980    0.9980    259335

PR-AUC: 0.8954375025034168


In [79]:
joblib.dump(
    final_rf,
    "../models/random_forest_final.pkl"
)

['../models/random_forest_final.pkl']